![Logo Curso](../assets/LogoCurso_gemini.png)

# MÓDULO 07: Análisis Exploratorio de Datos (EDA) y Curación de Metadatos Científicos de Digital.CSIC

**Curso:** Python para Ciencia Abierta (CSIC)  
**Autor:** Gustavo Liñán Cembrano  
**Fechas:** 5 a 8 de Octubre de 2026  
**Tiempo Estimado de Impartición:** 1 hora 30 minutos  
**Módulo:** 07 - Exploratory Data Analysis (EDA), OAI-PMH Scraping & Data Curated Pipelines  

---

## 1. Objetivos de Aprendizaje

- **Ingesta de Datos en Repositorios Abiertos:** Muestrear y descargar metadatos reales desde el protocolo OAI-PMH de **Digital.CSIC** utilizando la librería `digital_csic`.

- **Cortesía de Red (Throttling):** Respetar la carga del servidor aplicando pausas deliberadas entre peticiones HTTP (`delay = 1.0`s por registro).

- **Simulación de Corrupción de Datos:** Comprender los errores típicos en bases de datos científicas mediante la inyección controlada de anómalos (fechas de 5 dígitos por multiplicación, handles inexistentes, duplicados y nulos).

- **Análisis Exploratorio de Datos (EDA):** Dominar técnicas de inspección visual y estadística de calidad de datos (`shape`, `head`, `tail`, `info`, `describe`).

- **Pipelines de Limpieza y Curación:** 
  - Desduplicación por identificadores primarios (`drop_duplicates`).

  - **Validación de Identificadores por Consulta al Repositorio:** Verificar la validez de los Handles comprobando la existencia real del registro en Digital.CSIC mediante su API/OAI-PMH en lugar de depender únicamente de expresiones regulares cambiantes.

  - Detección y corrección de fechas fuera de rango (detección de años de más de 4 dígitos).

  - Imputación y normalización de textos y metadatos incompletos.
  
- **Publicación de Datasets Curados:** Exportar el conjunto de datos resultante en formatos estándar de Ciencia Abierta.


## 2. Configuración e Importación de Librerías

Importamos los módulos necesarios para el análisis de datos (`pandas`, `numpy`), generación de números aleatorios (`random`) e integración con la librería propia `digital_csic`.

> 💡 **Formato de Importación Estándar:** Usamos el alias `dcsic` para nuestra librería personalizada colocada en el directorio `lib/`.


In [ ]:
import sys
import os
import random
import pandas as pd
import numpy as np

### 2.1. Resolución Dinámica de Rutas y Configuración del Entorno (`sys.path`)

Cuando organizamos un proyecto de software científico en subcarpetas (por ejemplo, ejecutando cuadernos interactivos dentro del directorio `NOTEBOOKS/` mientras las librerías auxiliares residen en `lib/`), el intérprete de Python busca por defecto los módulos importables únicamente en el directorio actual de ejecución.

Esta celda garantiza la **portabilidad y reproducibilidad** del cuaderno independientemente de si se lanza desde la raíz del repositorio o desde la subcarpeta `NOTEBOOKS/`:

1. **Detección del Entorno de Ejecución:**
   Comprueba mediante `os.path.exists("../lib")` si la carpeta de librerías se encuentra en el nivel superior (ejecución habitual desde `NOTEBOOKS/`). Si existe, establece como `root_path` la ruta absoluta del directorio padre (`..`); en caso contrario, asume que el cuaderno se está ejecutando desde la raíz (`.`).

2. **Inclusión en `sys.path`:**
   Verifica si `root_path` ya está registrado en `sys.path` (la lista de rutas del sistema donde Python localiza paquetes y módulos). Si no lo está, lo añade dinámicamente con `sys.path.append(root_path)`, permitiendo importar directamente paquetes locales como `lib.digital_csic` o `lib.mylib` sin errores de tipo `ModuleNotFoundError`.


In [ ]:
# Añadir el directorio raíz del proyecto a sys.path para acceder al paquete lib/
root_path = os.path.abspath("..") if os.path.exists("../lib") else os.path.abspath(".")
if root_path not in sys.path:
    sys.path.append(root_path)


In [5]:

import lib.digital_csic as dcsic
from lib.mylib import linea as linea
# Ruta base flexible para el directorio DATASETS
datasets_dir = "../DATASETS" if os.path.exists("../DATASETS") else "DATASETS"
linea()
print(f"Versión de Pandas: {pd.__version__}")
print(f"Librería digital_csic cargada correctamente.")
print(f"Directorio de datos activo: {datasets_dir}")
linea()


****************************************************************************************************
Versión de Pandas: 3.0.5
Librería digital_csic cargada correctamente.
Directorio de datos activo: ../DATASETS
****************************************************************************************************


## **NOTA PARA EL ALUMNADO (PUNTOS 3 A 5):**
 
 El código contenido en los puntos 3 a 5 se ejecuta exclusivamente para **generar programáticamente una base de datos corrupta** a partir de registros reales de *Digital.CSIC*. 

 * **No os preocupéis por la complejidad de este código generador:** Es contenido avanzado fuera del alcance del curso.

 * **Su único propósito:** Crear de forma automática un conjunto de datos "sucio" (con nulos, tipados erróneos, fechas incoherentes y duplicados) para que veamos las técnicas de exploración y limpieza con Pandas en este módulo.


## 3. Muestreo e Ingesta de Datos Científicos desde Digital.CSIC

Cargaremos la lista masiva de identificadores Handle desde `DATASETS/FROM_DIGITALCSIC/input_handles.csv`, seleccionaremos una **muestra aleatoria de 100 registros** y consultaremos el API OAI-PMH de Digital.CSIC aplicando **1 segundo de pausa entre peticiones** para no saturar los servidores del CSIC.


In [6]:
# Cargar lista completa de handles disponibles
handles_file = os.path.join(datasets_dir, "FROM_DIGITALCSIC", "input_handles.csv")

all_handles = dcsic.load_handles_from_file(handles_file)

NREGS=50

linea()
print(f"Total de Handles disponibles en input_handles.csv: {len(all_handles)}")

# Fijar semilla aleatoria para reproducibilidad pedagógica
random.seed(42)
sample_handles = random.sample(all_handles, NREGS)

print(f"Muestra de {NREGS} Handles seleccionados aleatoriamente.")
print("Primeros 5 Handles de la muestra:", sample_handles[:5])
linea()

16:28:48 [INFO] Cargados 11310 Handles válidos desde '../DATASETS/FROM_DIGITALCSIC/input_handles.csv'


****************************************************************************************************
Total de Handles disponibles en input_handles.csv: 11310
Muestra de 50 Handles seleccionados aleatoriamente.
Primeros 5 Handles de la muestra: ['10261/398193', '10261/246719', '10261/158705', '10261/329833', '10261/309329']
****************************************************************************************************


In [7]:
# Descarga programática de metadatos desde Digital.CSIC
# IMPORTANTE: Aplicamos delay=1.0 segundo para cortesía de red
print("Iniciando la descarga de metadatos OAI-PMH (con 1.0s de pausa por cortesía)...")
df_raw = dcsic.fetch_records_batch(sample_handles, max_workers=4, delay=1.0, show_progress=True)

linea()
print("Dimensiones del dataset descargado:", df_raw.shape)
print("Primeras filas obtenidas:")
print(df_raw[['handle', 'title', 'year', 'journal']].head())
linea()

# Guardar copia del dataset original limpio
raw_output_path = os.path.join(datasets_dir, "digital_csic_raw.csv")
df_raw.to_csv(raw_output_path, index=False)
print(f"Dataset original guardado en '{raw_output_path}'")

16:29:55 [INFO] Iniciando procesamiento masivo de 50 registros de Digital.CSIC (4 hilos)...


Iniciando la descarga de metadatos OAI-PMH (con 1.0s de pausa por cortesía)...


Digital.CSIC: 100%|██████████| 50/50 [00:39<00:00,  1.26it/s]
16:30:35 [INFO] Procesamiento finalizado. Registros recuperados exitosamente: 45/50


****************************************************************************************************
Dimensiones del dataset descargado: (45, 8)
Primeras filas obtenidas:
         handle                                              title  year  \
0  10261/329833  El Archivo del MNCN participa en las VI Jornad...  2023   
1  10261/398193  Recurrence of the Spatial Structure of Summer ...  2025   
2  10261/246719    Challenge 3: Achieving a Resilient Living Ocean  2021   
3  10261/158705  Hindcasting the dynamics of an Eastern Mediter...  2018   
4  10261/305656  Pilot study to determine the association betwe...  2023   

                                             journal  
0  CSIC - Unidad de Recursos de Información Cient...  
1                                  John Wiley & Sons  
2                                     Editorial CSIC  
3                                     Inter Research  
4                                           Elsevier  
******************************************

## 4. Inyección Controlada de Errores (Corrupción del Dataset)

En la práctica científica real, los datos frecuentemente sufren corrupciones durante la recolección, formateo o migración.

Para aprender a detectar y reparar estos fallos, inyectaremos **4 tipos de errores aleatorios** sobre una copia de nuestro dataset:

1. **Fechas Erróneas (5 dígitos):** Multiplicar 5 años aleatorios por 10 (ej. `2021` --> `20210`).
2. **Filas Duplicadas:** Duplicar aleatoriamente 5 registros mediante concatenación.
3. **Handles Inexistentes:** Sustituir 5 Handles por identificadores inexistentes en el servidor (ej. `10261/99999991`).
4. **Resúmenes / Títulos Nulos:** Inserción de 5 valores faltantes (`NaN` / `None`).


In [8]:
# Copia de trabajo para la corrupción de datos
df_corrupted = df_raw.copy()
np.random.seed(42)


In [9]:

# 1. Corrupción de Fechas (Multiplicar 5 años por 10 provocando 5 dígitos)
indices_fecha = np.random.choice(df_corrupted.index, size=5, replace=False)
print("Indices de registros a los que corremperemos la fecha:", indices_fecha)
linea()
for idx in indices_fecha:
    val = df_corrupted.loc[idx, 'year']
    if str(val).isdigit():
        df_corrupted.loc[idx, 'year'] = str(int(val) * 10)
print(f"1. Inyectadas 5 fechas erróneas (años de 5 dígitos) en índices: {(indices_fecha)}")

Indices de registros a los que corremperemos la fecha: [39 25 26 43 35]
****************************************************************************************************
1. Inyectadas 5 fechas erróneas (años de 5 dígitos) en índices: [39 25 26 43 35]


In [10]:
# 2. Inyección de Duplicados (Duplicar 5 filas aleatorias)
indices_dup = np.random.choice(df_corrupted.index, size=5, replace=False)
filas_duplicadas = df_corrupted.loc[indices_dup]
df_corrupted = pd.concat([df_corrupted, filas_duplicadas], ignore_index=True)

linea()
print(f"2. Duplicadas 5 filas aleatorias. Nuevo tamaño del dataset: {df_corrupted.shape[0]}")

****************************************************************************************************
2. Duplicadas 5 filas aleatorias. Nuevo tamaño del dataset: 50


In [11]:
# 3. Corrupción de Handles (Sustituir por identificadores inexistentes en Digital.CSIC)
indices_handle = np.random.choice(df_corrupted.index, size=5, replace=False)
for i, idx in enumerate(indices_handle):
    df_corrupted.loc[idx, 'handle'] = f"10261/9999999{i}"

linea()
print(f"3. Inyectados 5 Handles inexistentesen índices: {(indices_handle)}.\nNuevo tamaño del dataset: {df_corrupted.shape[0]}")

****************************************************************************************************
3. Inyectados 5 Handles inexistentesen índices: [26  3 35 29 31].
Nuevo tamaño del dataset: 50


In [12]:
# 4. Inyección de Valores Faltantes (NaN en abstract)
indices_null = np.random.choice(df_corrupted.index, size=5, replace=False)
for idx in indices_null:
    df_corrupted.loc[idx, 'abstract'] = None

linea()
print(f"4. Inyectados 5 valores nulos en 'abstract' en índices: {(indices_null)}")



****************************************************************************************************
4. Inyectados 5 valores nulos en 'abstract' en índices: [42 13 23 39 17]


In [13]:
# 5. Inyección de Valores Faltantes (NaN en Titulo)
indices_null = np.random.choice(df_corrupted.index, size=5, replace=False)
for idx in indices_null:
    df_corrupted.loc[idx, 'title'] = None

linea()
print(f"5. Inyectados 5 valores nulos en 'title' en índices: {(indices_null)}")


****************************************************************************************************
5. Inyectados 5 valores nulos en 'title' en índices: [12 15 37 33  6]


In [14]:
# Guardar el dataset corrompido en disco
corrupted_output_path = os.path.join(datasets_dir, "digital_csic_corrupted.csv")
df_corrupted.to_csv(corrupted_output_path, index=False)
print(f"Dataset corrompido guardado en '{corrupted_output_path}'")
linea()

Dataset corrompido guardado en '../DATASETS/digital_csic_corrupted.csv'
****************************************************************************************************


## 5. Análisis Exploratorio de Datos (EDA) e Inspección Inicial

Comenzamos la fase de diagnóstico sobre el dataset que hemos corrompido `digital_csic_corrupted.csv` utilizando las funciones fundamentales de exploración de datos en Pandas.

---

In [15]:
# Cargar el dataset corrompido para iniciar la curación
corrupted_file = os.path.join(datasets_dir, "digital_csic_corrupted.csv")
df = pd.read_csv(corrupted_file)
linea('-',10)

----------


In [16]:

print(f"1. Tamaño y forma del dataset (.shape): {df.shape[0]} filas, {df.shape[1]} columnas")
linea()
print("2. Primeras 3 filas (.head(3)):")
print(df[['handle', 'year', 'title']].head(3))
linea()
print("3. Últimas 3 filas (.tail(3)):")
print(df[['handle', 'year', 'title']].tail(3))
linea()

1. Tamaño y forma del dataset (.shape): 50 filas, 8 columnas
****************************************************************************************************
2. Primeras 3 filas (.head(3)):
         handle  year                                              title
0  10261/329833  2023  El Archivo del MNCN participa en las VI Jornad...
1  10261/398193  2025  Recurrence of the Spatial Structure of Summer ...
2  10261/246719  2021    Challenge 3: Achieving a Resilient Living Ocean
****************************************************************************************************
3. Últimas 3 filas (.tail(3)):
          handle   year                                              title
47  10261/127040  20150                Vino, fuente natural de polifenoles
48  10261/362005   2024  Belowground biodiversity is essential for supp...
49  10261/164389   2018  Un perfil de las personas mayores en España, 2...
**********************************************************************************

In [18]:
# Resumen técnico (.info() y .describe())
linea()
print("Información de tipos y nulos (.info()):")
df.info()
linea()


****************************************************************************************************
Información de tipos y nulos (.info()):
<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   handle    50 non-null     str  
 1   title     45 non-null     str  
 2   abstract  45 non-null     str  
 3   authors   50 non-null     str  
 4   year      50 non-null     int64
 5   journal   50 non-null     str  
 6   doi       50 non-null     str  
 7   source    50 non-null     str  
dtypes: int64(1), str(7)
memory usage: 86.0 KB
****************************************************************************************************


In [19]:

print("Resumen estadístico de columnas (.describe()):")
print(df.describe(include='all'))
linea()

Resumen estadístico de columnas (.describe()):
              handle                                              title  \
count             50                                                 45   
unique            46                                                 41   
top     10261/329833  El Archivo del MNCN participa en las VI Jornad...   
freq               2                                                  2   
mean             NaN                                                NaN   
std              NaN                                                NaN   
min              NaN                                                NaN   
25%              NaN                                                NaN   
50%              NaN                                                NaN   
75%              NaN                                                NaN   
max              NaN                                                NaN   

                                                 abs

## 6. Bloque de Limpieza y Curación de Datos

Procederemos a solucionar metódicamente cada una de las anomalías detectadas en nuestro dataset científico.


### 6.1. Detección y Eliminación de Duplicados

In [20]:
# Identificar duplicados basados en la columna 'handle'
n_duplicados = df.duplicated(subset=['handle']).sum()

linea()
print(f"Registros totales antes de desduplicar: {df.shape[0]}")
print(f"Número de registros duplicados por Handle: {n_duplicados}")

# Eliminar duplicados manteniendo la primera aparición
df_clean = df.drop_duplicates(subset=['handle'], keep='first').copy()

print(f"Registros tras eliminar duplicados: {df_clean.shape[0]}")
linea()

****************************************************************************************************
Registros totales antes de desduplicar: 50
Número de registros duplicados por Handle: 4
Registros tras eliminar duplicados: 46
****************************************************************************************************


### 6.2. Validación de Handles: Verificación Real frente al Repositorio vs Expresiones Regulares

> 🎓 **Lección Clave de Ciencia Abierta:**  
> A menudo se utiliza la validación por **expresiones regulares (Regex)** para comprobar la sintaxis de un identificador (ej. buscar el patrón `10261/XXXXX`). Sin embargo, **depender únicamente del formato es un error común en producción**: los formatos de URIs, prefijos institucionales o identificadores persistentes pueden evolucionar con el tiempo.
> 
> La forma verdaderamente robusta de validar la integridad de un Handle o DOI es **verificar la existencia real del registro consultando directamente el servicio OAI-PMH de Digital.CSIC**. Si el servidor no devuelve metadatos o indica error de registro no encontrado (`idDoesNotExist`), sabremos con certeza que el identificador es inválido o corrupto.


### 6.3 Creamos una pequeña funcion que devuelve si un registros existe o no en Digital.CSIC mediante consulta al OAI.PMH

In [21]:
# Función de validación basada en verificación real con el API de Digital.CSIC
def verificar_existencia_handle(handle_str):
    # Consultamos al servidor de Digital.CSIC si el registro existe realmente
    record = dcsic.fetch_digital_csic_record(handle_str, enrich_external=False)
    return record is not None


In [22]:

linea()
print("Iniciando la validación de Handles mediante consulta directa al repositorio Digital.CSIC...")

# Evaluamos cada handle sobre el DataFrame cargado desde el CSV corrompido. 
# Para ello creamos una columna "valido" que resulta de aplicar la funcion verificar_existencia_handle
# a la columna handle

df_clean['valido'] = df_clean['handle'].apply(verificar_existencia_handle)

print(f"Registros tras verificar handle: {df_clean.shape[0]}")
linea()

****************************************************************************************************
Iniciando la validación de Handles mediante consulta directa al repositorio Digital.CSIC...


16:33:42 [WARNING] OAI-PMH retornó error para el handle 10261/99999991
16:33:47 [WARNING] OAI-PMH retornó error para el handle 10261/99999990
16:33:48 [WARNING] OAI-PMH retornó error para el handle 10261/99999993
16:33:48 [WARNING] OAI-PMH retornó error para el handle 10261/99999994
16:33:49 [WARNING] OAI-PMH retornó error para el handle 10261/99999992


Registros tras verificar handle: 46
****************************************************************************************************


🧠 ¿Qué hace exactamente el método .apply()?

El método .apply() en Pandas es como un bucle for automático, elegante y optimizado.

Su función es tomar una ** Serie de datos** (una columna) y pasar elemento por elemento (celda a celda) por una función de Python que tú elijas.


In [23]:

# Filtrar handles inválidos (aquellos que el servidor rechazó)
handles_invalidos = df_clean[~df_clean['valido']]
print(f"Número de Handles no encontrados o inválidos en el servidor: {len(handles_invalidos)}")
print("Handles defectuosos detectados por la consulta real:")
print(handles_invalidos[['handle', 'title']])
linea()


Número de Handles no encontrados o inválidos en el servidor: 5
Handles defectuosos detectados por la consulta real:
            handle                                              title
3   10261/99999991  Hindcasting the dynamics of an Eastern Mediter...
26  10261/99999990                Vino, fuente natural de polifenoles
29  10261/99999993  A radiation of Psylliodes flea beetles on Bras...
31  10261/99999994  Formation and structures of horizontal submari...
35  10261/99999992  Advances on biomass-derived microporous carbon...
****************************************************************************************************


In [24]:

# Conservar únicamente los registros validados por el servidor
df_clean = df_clean[df_clean['valido']].drop(columns=['valido']).copy()
print(f"Registros totales tras filtrar Handles inválidos: {df_clean.shape[0]}")
linea()

Registros totales tras filtrar Handles inválidos: 41
****************************************************************************************************


### 6.3. Detección y Corrección de Fechas Anómalas (Búsqueda de Años de Más de 4 Dígitos)

En bases de datos temporales, los años válidos constan de **4 dígitos** (entre 1000 y 9999). Cualquier valor con 5 o más dígitos (o `year_num >= 10000`) refleja una corrupción de datos.


In [25]:
# Convertir columna year a numérico con errors='coerce'
df_clean['year_num'] = pd.to_numeric(df_clean['year'], errors='coerce')

# Detectar fechas anómalas buscando años con más de 4 dígitos (>= 10000)
mask_mas_de_4_digitos = df_clean['year_num'] >= 10000
fechas_erroneas = df_clean[mask_mas_de_4_digitos]

linea()
print(f"Número de fechas anómalas detectadas (más de 4 dígitos): {len(fechas_erroneas)}")
print("Muestra de fechas erróneas de 5 dígitos encontradas:")
print(fechas_erroneas[['handle', 'year', 'year_num']])
linea()

# Corrección de fechas de 5 dígitos (división entera por 10 para restaurar el año de 4 dígitos)
df_clean.loc[mask_mas_de_4_digitos, 'year_num'] = df_clean.loc[mask_mas_de_4_digitos, 'year_num'] // 10

# Formatear como entero definitivo
df_clean['year'] = df_clean['year_num'].fillna(0).astype(int)
df_clean.drop(columns=['year_num'], inplace=True)

print("Fechas corregidas exitosamente. Resumen estadístico de 'year':")
print(f"Maxima fecha encontrada: {df_clean['year'].describe()}")
linea()

****************************************************************************************************
Número de fechas anómalas detectadas (más de 4 dígitos): 4
Muestra de fechas erróneas de 5 dígitos encontradas:
          handle   year  year_num
25  10261/330088  20230     20230
39  10261/342915  20240     20240
43  10261/383282  20250     20250
47  10261/127040  20150     20150
****************************************************************************************************
Fechas corregidas exitosamente. Resumen estadístico de 'year':
Maxima fecha encontrada: count      41.000000
mean     2022.658537
std         2.424972
min      2015.000000
25%      2021.000000
50%      2023.000000
75%      2025.000000
max      2025.000000
Name: year, dtype: float64
****************************************************************************************************


### 6.4. Tratamiento de Valores Faltantes (NaN) e Imputación

In [26]:
# Conteo de valores nulos por columna
nulos_por_columna = df_clean.isnull().sum()

linea()
print("Recuento de valores nulos por columna:")
print(nulos_por_columna)
linea()


****************************************************************************************************
Recuento de valores nulos por columna:
handle      0
title       5
abstract    5
authors     0
year        0
journal     0
doi         0
source      0
dtype: int64
****************************************************************************************************


In [28]:
# Asignacion de abstract nulos con valor por defecto
df_clean.fillna({
    'abstract': 'Sin abstract disponible',
    'journal': 'Digital.CSIC',
    'authors': 'Autor Desconocido'
}, inplace=True)

print("Nulos restantes tras la imputación:")
print(df_clean.isnull().sum())
linea()

Nulos restantes tras la imputación:
handle      0
title       5
abstract    0
authors     0
year        0
journal     0
doi         0
source      0
dtype: int64
****************************************************************************************************


In [29]:
# Asignacion de título nulos con valor por defecto
df_clean.fillna({
    'title': 'Sin título disponible',
    'journal': 'Digital.CSIC',
    'authors': 'Autor Desconocido'
}, inplace=True)

print("Nulos restantes tras la imputación:")
print(df_clean.isnull().sum())
linea()

Nulos restantes tras la imputación:
handle      0
title       0
abstract    0
authors     0
year        0
journal     0
doi         0
source      0
dtype: int64
****************************************************************************************************


## 7. Análisis Exploratorio Final (EDA Curado)

Una vez curado el dataset, realizamos el análisis exploratorio definitivo sobre la distribución temporal y temática de la producción científica.


In [30]:
# Distribución de publicaciones por década / año
publicaciones_por_año = df_clean[df_clean['year'] > 1900]['year'].value_counts().sort_index()

linea()
print("Distribución de publicaciones por año (Top 10 años más recientes):")
print(publicaciones_por_año.tail(10))
linea()

# Top 5 revistas más frecuentes en la muestra
top_revistas = df_clean['journal'].value_counts().head(5)
print("Top 5 Fuentes / Revistas más frecuentes:")
print(top_revistas)
linea()

****************************************************************************************************
Distribución de publicaciones por año (Top 10 años más recientes):
year
2015     1
2018     3
2019     1
2020     1
2021     6
2022     1
2023    10
2024     7
2025    11
Name: count, dtype: int64
****************************************************************************************************
Top 5 Fuentes / Revistas más frecuentes:
journal
Elsevier                   12
Frontiers Media             3
John Wiley & Sons           2
Springer                    2
Nature Publishing Group     2
Name: count, dtype: int64
****************************************************************************************************


## 8. Exportación del Dataset Curado de Ciencia Abierta

Exportamos el dataset final 100% limpio y curado a un archivo CSV estructurado listo para su publicación e interoperabilidad.


In [31]:
# Exportar a CSV con codificación UTF-8
output_curated = os.path.join(datasets_dir, "digital_csic_curated.csv")
dcsic.export_records_to_csv(df_clean, output_curated, index=False)

linea()
print(f"Dataset curado exportado exitosamente a: '{output_curated}'")
print(f"Registros totales finales: {df_clean.shape[0]}")
linea()

16:35:29 [INFO] Archivo CSV guardado exitosamente en: /Users/linan/Desktop/RESEARCH/FORMACION/PYTHON_PARA_CIENCIA_ABIERTA_2/DATASETS/digital_csic_curated.csv


****************************************************************************************************
Dataset curado exportado exitosamente a: '../DATASETS/digital_csic_curated.csv'
Registros totales finales: 41
****************************************************************************************************


## 9. Ejercicios Prácticos y Autoevaluación

### Ejercicio 1: Filtrado de Producción Reciente
Filtra el DataFrame curado `df_clean` para mostrar únicamente los registros publicados a partir del año **2020** inclusive. Muestra el número total de artículos y las columnas `handle`, `year` y `title`.


In [ ]:
# Solución Ejercicio 1
recientes = df_clean[df_clean['year'] >= 2020]

linea()
print(f"Número de artículos publicados a partir de 2020: {len(recientes)}")
print(recientes[['handle', 'year', 'title']].head())
linea()

### Ejercicio 2: Búsqueda Palabras Clave en Títulos
Encuentra cuántos artículos contienen la palabra `"investigación"` o `"estudio"` en su título (insensible a mayúsculas/minúsculas).


In [ ]:
# Solución Ejercicio 2
mask_kw = df_clean['title'].str.contains(r'investigación|estudio|research|study', case=False, na=False)
articulos_kw = df_clean[mask_kw]

linea()
print(f"Número de artículos con palabras clave científicas en el título: {len(articulos_kw)}")
print(articulos_kw[['handle', 'title']].head())
linea()

---

### 🎬 Videopodcast Resumen del Módulo (07)

<div align="center">
  <video width="100%" style="max-width: 800px; border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.15);" controls>
    <source src="../assets/VIDEOPODCASTS/modulo_07.mp4" type="video/mp4">
    Tu navegador o entorno de Jupyter no soporta reproducción directa de vídeos HTML5.
  </video>
</div>

---

### 📝 Cuestionario de Auto-evaluación y Participación

¡Pon a prueba los conocimientos adquiridos en este módulo! Completa este breve cuestionario interactivo de 5 minutos en Google Forms:

👉 **[Cuestionario Módulo 07 en Google Forms](https://docs.google.com/forms/d/e/1FAIpQLSe9z1C-MrA93NAZWvbprrpHGiI0t5JAoPW_5LqlvKB5o33F_Q/viewform?usp=sharing&ouid=101670006760862697598)**

---

### 🎧 Tutor Virtual & Libreta de Estudio (Google NotebookLM)

¿Quieres escuchar el resumen conversacional en audio de este módulo o resolver dudas con el tutor virtual del curso?

👉 **[Acceder al Espacio de Estudio Interactivo en NotebookLM](https://notebook.google.com/notebook/d76ad7db-e258-4386-a050-9c15d29d8fb2/artifact/08aec159-c0c7-44df-90d1-2d29666a79f1)**

---
